# E12 — cắt ngẫu nhiên từ cache có lề dư, thay cho tịnh tiến-đệm-0

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

## Lỗi mà thí nghiệm này sửa

`RandomTranslate3D` dịch ảnh rồi **đệm 0**; `RandomRotateSmall` lấp góc bằng 0
(`cval=0.0`, prob 1.0). Đo được:

> **≈100% mẫu TRAIN mang một dải đen ở rìa. 0% mẫu VAL có nó.**

Đây là lệch phân bố train/val có hệ thống, xuất hiện ở **mọi bước huấn luyện**, và
chưa từng được nhận ra qua 12 thí nghiệm. Nó khớp với chẩn đoán overfit đã đo:
`val_loss` chạm đáy sớm, tương quan hạng **ρ = +0,770 (P = 0,0092)** với macro-F1
cuối cùng của fold (WORKLOG S-107).

Baseline official và CGHNet đều **không** làm vậy — họ cache rộng hơn rồi cắt ngẫu
nhiên (official resize 128² cắt 112²; CGHNet 16×128×128 → 14×112×112). Ablation
CGHNet Bảng 4: **bỏ random-crop mất 8,8 điểm**, nặng nhất trong bảng của họ.

## Cách sửa

| | cũ (E4) | mới (E12) |
|---|---|---|
| lưới cache | 112×112×32 | **136×136×40** |
| model nhận | 112×112×32 | 112×112×32 |
| lúc train | dịch ±8, đệm 0 | **cắt ngẫu nhiên ±12, mô thật** |
| lúc val | không dịch | **cắt giữa** |
| xoay lấp góc bằng | 0 | **nhân bản voxel biên** |

**Tính chất quan trọng:** `spacing` vẫn suy từ 112×112×32, không từ lưới 136×136×40.
Nên độ phân giải vật lý y hệt E4, và **cắt giữa cache E12 cho ra đúng khối mà cache
E4 tạo ra**. Val của hai bên so trực tiếp được ⇒ phép so E12–E4 chỉ khác **một** biến.

⚠️ Lề 12 chứ không phải 8: xoay 10° trên khối 136 làm hỏng góc tới ~12 voxel. Đo
thật: với `mode=constant`, cắt ở offset biên để lọt **517** voxel bị lấp 0; đổi sang
`mode=nearest` thì **0** voxel ở mọi offset. Cả hai điều chỉnh đều cần thiết.

## Cách đọc kết quả — chốt TRƯỚC khi chạy

Mốc E4, cùng bệnh nhân, cùng fold:

    fold 1 0.7001 | 2 0.6771 | 3 0.7304 | 4 0.6680 | 5 0.6618 | gộp 394 ca 0.6851
    Cột `last` (KHÔNG thiên lệch) của E4: 0.6038  <- mốc thật để so

| quan sát | kết luận |
|---|---|
| CI95 của hiệu không chứa 0, dương | E12 thành cấu hình gốc mới |
| CI95 chứa 0 | giữ E4 làm gốc, nhưng **vẫn giữ E12** vì dải đệm là lỗi thật |
| khoảng cách `best − last` hẹp lại | đã chữa đúng phần overfit nhắm tới |
| epoch chạm đáy `val_loss` muộn hơn | bằng chứng độc lập, xem ρ = 0,770 |

⚠️ **2 fold không đủ để chốt.** E6b sàng 2 fold cho +0,038 rồi 5 fold cho −0,002.
Notebook này mặc định chạy 3 fold mỗi session; chạy đủ 5 rồi mới kết luận.

## Cần mount

**Dữ liệu gốc LLD-MMRI** (để build cache mới). Cache E4 cũ **không dùng được** —
nó không có lề dư, và cell train sẽ nổ ngay với thông báo nói rõ lý do.

## 0. Bootstrap

Giống notebook 07. Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3]              # 3 fold/session; chạy [4, 5] ở session sau
CONFIG_NAME = "e12_randomcrop.yaml"
PREPROCESS_NAME = "preprocess_e12.yaml"
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
PRE_PATH = REPO / "configs" / PREPROCESS_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(PRE_PATH)

INNER = tuple(CFG["data"]["crop_size"])
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))
assert tuple(PRE["target_size"]) == INNER, "hai config lệch nhau"

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"lưới cache: {GRID}  ->  model nhận: {INNER}  (lề {MARGIN})")
print(f"xoay lấp góc bằng: {CFG['data']['augment'].get('rotate_mode', 'constant')}")

## Cổng 0 ⚠️ — config này khác baseline ở ĐÚNG những chỗ nào

Bốn khoá, không hơn. Thừa một khoá là thí nghiệm hai biến, và lỗi đó không để lại
dấu vết nào trong kết quả.

In [ ]:
BASE = load_yaml(REPO / "configs" / "baseline_3dpatch.yaml")

def flat(d, p=""):
    out = {}
    for k, v in d.items():
        n = f"{p}{k}"
        out.update(flat(v, n + ".")) if isinstance(v, dict) else out.update({n: v})
    return out

MIEN = {"cache_dir", "output_dir"}   # đường dẫn, không phải biến khoa học
CHO_PHEP = {"data.crop_size", "data.augment.translate_voxels", "data.augment.rotate_mode"}

a, b = flat(BASE), flat(CFG)
khac = {k for k in set(a) | set(b) if a.get(k) != b.get(k)} - MIEN
print("khác baseline:")
for k in sorted(khac):
    print(f"  {k}: {a.get(k)!r} -> {b.get(k)!r}")

ngoai = khac - CHO_PHEP
assert not ngoai, f"⛔ có khoá NGOÀI phạm vi cho phép: {ngoai}"
assert khac == CHO_PHEP, f"⛔ thiếu khoá mong đợi: {CHO_PHEP - khac}"
print("\n✓ đúng 3 khoá khoa học + đường dẫn")

## 2. Build cache có lề dư

Không dùng lại được cache E4: nó là 112×112×32, không có lề. Build mới mất khoảng
**45 phút** (lớn hơn E4 1,84 lần vì lưới 136×136×40).

Cache ghi vào `/kaggle/working/cache_e12`. Sau khi chạy xong **nhớ tải về và upload
thành Kaggle Dataset**, nếu không thì session sau phải build lại.

In [ ]:
import json as _json
import time

from src.utils.io import resolve_data_root

CACHE_DIR = Path("/kaggle/working/cache_e12")
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

# Cache đã mount sẵn ở đâu đó chưa? Nhận diện bằng NỘI DUNG cache_meta.json, không
# bằng tên dataset — tên do người upload đặt và đã lệch một lần rồi (S-080).
E12_KEYS = {
    "align_phases": "per_phase",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
    "crop_mode": "lesion_tight",
}
san_co = None
for meta_path in Path("/kaggle/input").rglob("cache_meta.json"):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:
        continue
    if all(meta.get(k) == v for k, v in E12_KEYS.items()):
        san_co = meta_path.parent
        break

if san_co is not None:
    n = len(list(san_co.glob("*.npz")))
    print(f"✓ đã có cache E12 mount sẵn: {san_co} ({n} ca) — bỏ qua build")
    CACHE_DIR = san_co
    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
else:
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None
    # `resolve_data_root` trả về config['data_root'] mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py). Trên Kaggle nó sẽ là đường dẫn tương đối không
    # tồn tại, và job 45 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        )
    print("data root:", data_root, "✓")
    t0 = time.time()
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", f"configs/{PREPROCESS_NAME}"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print(f"build xong sau {(time.time() - t0) / 60:.0f} phút: {CACHE_DIR}")

## Cổng A ⚠️ — cache có đúng lề dư không

Ba thứ phải đúng, và mỗi thứ chặn một lỗi đã từng xảy ra:

1. `cache_meta.json` khớp E12 — chặn việc lấy nhầm cache E4 cũ.
2. Hình dạng mảng thật đúng 136×136×40 — chặn việc `crop_margin_voxels` bị bỏ qua.
3. **Cắt giữa cache E12 phải cho ra khối 112×112×32** — đây là tính chất khiến val
   so được với E4.

In [ ]:
import numpy as np

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
for k, v in E12_KEYS.items():
    assert meta.get(k) == v, f"cache SAI: {k} = {meta.get(k)!r}, cần {v!r}"

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"

mau = next(CACHE_DIR.glob("*.npz"))
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
    assert shape == (8, *GRID), f"hình dạng {shape}, cần {(8, *GRID)}"
    assert tuple(z["crop_margin_voxels"]) == MARGIN
    assert tuple(z["inner_size"]) == INNER

# Cắt giữa ra đúng kích thước model, và KHÔNG có voxel đệm.
from src.data.transforms import CenterCrop3D

with np.load(mau) as z:
    cut = CenterCrop3D(INNER)({"image": z["image"].astype(np.float32)})["image"]
assert tuple(cut.shape) == (8, *INNER), cut.shape
print(f"cache E12 ✓ · {n_npz} ca · mảng {shape} -> cắt giữa {tuple(cut.shape)}")
print(f"commit build: {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — dải đệm đã biến mất chưa

Đây là cổng quan trọng nhất của cả notebook, vì nó đo trực tiếp thứ E12 sinh ra để
sửa. So tỉ lệ voxel bằng 0 ở rìa giữa mẫu **train** và mẫu **val**.

Ở E4 hai con số này lệch hẳn nhau (train có dải đen, val không). Ở E12 chúng phải
**ngang nhau**. Nếu vẫn lệch thì có gì đó chưa đúng và **đừng chạy train**.

In [ ]:
import torch

from src.train.run import build_loaders

train_loader, val_loader, _ = build_loaders(CFG, FOLDS[0])

def ty_le_zero_o_ria(loader, n_batch=8, bien=6):
    """Tỉ lệ voxel bằng 0 trong vỏ ngoài `bien` voxel của khối."""
    tong, dem = 0.0, 0
    for i, batch in enumerate(loader):
        if i >= n_batch:
            break
        x = batch["image"]
        vo = torch.ones_like(x, dtype=torch.bool)
        vo[:, :, bien:-bien, bien:-bien, bien:-bien] = False
        tong += float((x[vo] == 0).float().mean())
        dem += 1
    return tong / max(dem, 1)

tr = ty_le_zero_o_ria(train_loader)
va = ty_le_zero_o_ria(val_loader)
print(f"tỉ lệ voxel 0 ở vỏ ngoài:  train {tr:.4f}   val {va:.4f}")
print(f"kích thước batch train: {tuple(next(iter(train_loader))['image'].shape)}")

assert abs(tr - va) < 0.02, (
    f"⛔ train {tr:.4f} so với val {va:.4f} — dải đệm VẪN CÒN. Đây đúng là thứ E12 "
    f"sinh ra để xoá; chạy train bây giờ là lãng phí session. Kiểm `crop_size` trong "
    f"config và `rotate_mode: nearest`."
)
print("\n✓ train và val cùng phân bố ở rìa — dải đệm đã hết")

## 3. Ngân sách

Đo thời gian một epoch thật rồi suy ra cả session. E4 mất 45,1 s/epoch, E6b 38,0.
Cắt ngẫu nhiên rẻ hơn tịnh tiến (chỉ là chỉ số), nhưng khối đọc từ đĩa lớn hơn
1,84 lần nên I/O nặng hơn.

In [ ]:
import time

model_probe = None
t0 = time.time()
n = 0
for batch in train_loader:
    n += 1
    if n >= 20:
        break
giay_moi_batch = (time.time() - t0) / n
so_batch = len(train_loader)
uoc_giay_epoch = giay_moi_batch * so_batch
gio_moi_fold = uoc_giay_epoch * int(CFG["train"]["epochs"]) / 3600

print(f"{so_batch} batch/epoch · {giay_moi_batch:.2f}s/batch (chỉ nạp dữ liệu)")
print(f"ước tính CHƯA tính GPU: {uoc_giay_epoch:.0f}s/epoch -> {gio_moi_fold:.1f}h/fold")
print(f"{len(FOLDS)} fold -> {gio_moi_fold * len(FOLDS):.1f}h")
print("\n(E4 thật: 3.76h/fold · E6b: 3.17h/fold. Con số trên chỉ đo phần CPU nạp dữ")
print(" liệu nên sẽ THẤP hơn thực tế — dùng để phát hiện chậm bất thường, không để")
print(" lập kế hoạch.)")
if gio_moi_fold * len(FOLDS) > 11.0:
    print("\n⚠ vượt 11h — bỏ bớt fold khỏi FOLDS")

## 4. Train

`resume: true` nên bị ngắt giữa chừng thì chạy lại đúng cell này, nó đọc tiếp từ
`last.pt`.

In [ ]:
from src.train.run import train

for fold in FOLDS:
    print(f"\n{'=' * 60}\nFOLD {fold}\n{'=' * 60}")
    train(CFG, fold=fold)

## 5. Kết quả session này

In [ ]:
OUT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
rows = []
for d in sorted(OUT.glob("fold*")):
    f = d / "metrics_best.json"
    if f.exists():
        m = _json.loads(f.read_text("utf-8"))
        rows.append((m["fold"], m["epoch"], m["macro_f1"], m["cohen_kappa"]))

E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}
print(f"{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}{'E4':>9}{'hiệu':>9}")
print("-" * 50)
for fold, ep, f1, kp in rows:
    d = f1 - E4[fold] if fold in E4 else float("nan")
    print(f"{fold:>5}{ep:>7}{f1:>11.4f}{kp:>9.4f}{E4.get(fold, 0):>9.4f}{d:>+9.4f}")

print("\nEpoch chạm đáy val_loss — dự báo gần trọn vẹn F1 cuối (ρ=0.770, S-107):")
import csv as _csv
for d in sorted(OUT.glob("fold*")):
    log = d / "train_log.csv"
    if log.exists():
        vl = [float(x["val_loss"]) for x in _csv.DictReader(open(log))]
        print(f"  {d.name}: đáy @ epoch {vl.index(min(vl)) + 1}")
print("  (E4: fold 1 @100 · 2 @79 · 3 @227 · 4 @3 · 5 @14. Muộn hơn = ít overfit hơn.)")

print("\n⚠ Chênh lệch từng fold là NHIỄU nếu nhìn riêng (CI mỗi fold ~±0.19).")
print("  Đừng kết luận trước khi đủ 5 fold và so cặp trên 394 ca ở local.")

## 6. Gói mang về

**Hai thứ**: kết quả train, và **cache E12** nếu vừa build (nếu không thì session sau
phải build lại 45 phút).

In [ ]:
import shutil

PACK = Path(f"/kaggle/working/{EXPERIMENT}_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "val_probs_last.npz", "metrics_best.json",
        "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT.glob("fold*")):
    dst = PACK / d.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.1f} MiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz và .pt bản thân là zip; trình giải nén bung
  đệ quy sẽ biến chúng thành thư mục và src.eval.* không thấy (đã dính, S-078).

⚠ NẾU VỪA BUILD CACHE: upload /kaggle/working/cache_e12 thành Kaggle Dataset (Private
  theo license dữ liệu) trước khi đóng session. Không thì fold 4-5 phải build lại.

Ở local, sau khi đủ 5 fold:
    python -m src.eval.run --run-dir runs/e12_randomcrop
""")